# MODI-SYNTH · Baseline Reader — Kaggle edition

**Runs unattended.** Set it going with *Save Version → Save & Run All (Commit)*, close the browser.

### One-time setup
1. **Add data:** right panel → *Add Input* → *Upload* → `colab_pack.zip` (7.6 MB, from `FYP-modi-script-data/`). Any dataset name works; the notebook finds it.
2. **Settings:** Accelerator = **GPU T4 x2**, Internet = **On** (MoDeTrans is pulled from Hugging Face).
3. *Save Version → Save & Run All (Commit)*.

### What changed from the Colab notebook, and why
The earlier design fed the model the whole patch and asked it to work out for itself where the
text lines were. It never managed: across three runs it spent 9,500+ updates predicting spaces
(the commonest character) and once diverged to `nan`.

Here each patch is first **cut into its text lines** and each line is placed in its own fixed band,
in reading order. The model's output row *k* is now text line *k* **by construction**, so it only
has to recognise characters along a line — not discover the page layout as well. Line detection is
a plain projection profile and succeeded on all 2,043 patches (1–7 lines each).

Everything else — alphabet, splits, loss, metrics — is unchanged, so results stay comparable.

In [ ]:
# Cell 1 - environment, GPU, and the wall-clock budget that keeps this inside Kaggle's 12 h limit
import os, sys, time, json, re, math, random, csv, shutil, zipfile, pathlib
T_START = time.time()
BUDGET_H = 10.5          # stop training after this many hours so the notebook always finishes & saves
def elapsed_h(): return (time.time() - T_START) / 3600
def budget_left(): return BUDGET_H - elapsed_h()

import torch
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Turn on the GPU: Settings -> Accelerator -> GPU T4 x2'
print('GPU:', torch.cuda.get_device_name(0))
print(f'budget: {BUDGET_H} h of training, then it stops and saves whatever it has')

In [ ]:
# Cell 2 - every setting in one place. Values follow the published recipes, not guesses:
#   UniLipi 2026 (closest work, covers MODI): one line, height 68, Adam lr 2e-4, weight decay 0.01
#   HTR-VT, Pattern Recognition 2025        : one line, 512x64, max lr 1e-3 + warmup + cosine, batch 64
#   FLAME Sanskrit 2026                     : one line, height 50, Adam, batch 4, gradient clip 5
#   HF/community CRNN practice              : height 60-96, lr 1e-4..6e-4, batch 2-8, cosine, patience ~20
# lr 3e-4 = HTR-VT's 1e-3 scaled from batch 64 down to batch 4, which also lands on UniLipi's 2e-4.
# Trained in FULL fp32 - mixed precision overflowed and silently killed an earlier run.
OUT = '/kaggle/working'
CFG = dict(
    run_name   = 'B1_seed42', seed = 42,

    strip_h = 64, strip_w = 6144, line_gap = 20,   # one row of text, as in every working system
    batch_size = 4, accum = 1,
    lr = 3e-4, weight_decay = 1e-2, warmup_steps = 1000,
    max_epochs = 400, patience = 25, min_delta = 0.05,

    use_lstm = True, use_count = True, use_aug = True, span_mask = True,
    count_weight = 0.4, num_workers = 2,
)
CFG['img_h'], CFG['img_w'] = CFG['strip_h'], CFG['strip_w']
RUN_DIR = f"{OUT}/runs/{CFG['run_name']}"
os.makedirs(RUN_DIR, exist_ok=True)
print('input strip:', CFG['img_h'], 'x', CFG['img_w'], '| run dir:', RUN_DIR)

In [ ]:
# Cell 3 - libraries Kaggle does not ship
!pip -q install datasets editdistance sacrebleu 2>&1 | tail -1
print('installed')

In [ ]:
# Cell 4 - find colab_pack.zip anywhere under /kaggle/input and unpack the metadata
DATA = '/kaggle/working/data'
os.makedirs(DATA, exist_ok=True)
need = ['splits.json', 'modetrans_text.csv', 'charset.json']

def find_metadata():
    for root, _, files in os.walk('/kaggle/input'):
        if all(n in files for n in need):          # already-unzipped folder
            return root
    for root, _, files in os.walk('/kaggle/input'):
        for f in files:
            if f.endswith('.zip'):
                zipfile.ZipFile(os.path.join(root, f)).extractall(DATA)
                for r2, _, f2 in os.walk(DATA):
                    if all(n in f2 for n in need):
                        return r2
    return None

META = find_metadata()
assert META, ('Could not find the metadata. Add colab_pack.zip as an input dataset '
              '(right panel -> Add Input -> Upload).')
print('metadata:', META, sorted(os.listdir(META))[:6])

In [ ]:
# Cell 5 - MoDeTrans images + our fixed splits and cleaned text
import pandas as pd
from datasets import load_dataset

ds = load_dataset('historyHulk/MoDeTrans', split='train')     # 2,043 rows
names  = list(ds['filename'])
splits = json.load(open(f'{META}/splits.json', encoding='utf-8'))['modetrans']['assignment']
text   = (pd.read_csv(f'{META}/modetrans_text.csv', keep_default_na=False)
            .set_index('image_id')['text_train'].to_dict())

assert len(names) == 2043 and set(names) == set(splits) == set(text), 'dataset / split mismatch'
SPLIT_IDS = {s: [n for n in names if splits[n] == s] for s in ['train', 'val', 'test']}
assert not (set(SPLIT_IDS['train']) & (set(SPLIT_IDS['val']) | set(SPLIT_IDS['test']))), 'leakage!'
print('splits:', {k: len(v) for k, v in SPLIT_IDS.items()})

In [ ]:
# Cell 6 - alphabet: index 0 is the CTC blank, then the 86 symbols, then <FRAC> and <unk>
cs = json.load(open(f'{META}/charset.json', encoding='utf-8'))
VOCAB = ['<blank>'] + cs['symbols'] + ['<FRAC>', '<unk>']
STOI = {c: i for i, c in enumerate(VOCAB)}
NUM_CLASSES = len(VOCAB)
_TOK = re.compile(r'<FRAC>|.', re.S)
encode     = lambda s: [STOI.get(t, STOI['<unk>']) for t in _TOK.findall(s)]
decode_ids = lambda ids: ''.join(VOCAB[i] for i in ids if i > 0)

bad = [n for n in names if decode_ids(encode(text[n])) != text[n]]
assert not bad, f'round-trip failed: {bad[:5]}'
assert sum(t == STOI['<unk>'] for n in SPLIT_IDS['train'] for t in encode(text[n])) == 0
MAX_LEN = max(len(encode(text[n])) for n in names)
print(f'{NUM_CLASSES} classes, longest label {MAX_LEN} characters')

In [ ]:
# Cell 7 - THE KEY STEP. Cut the patch into its text lines, crop each one to its ink,
# and lay them END TO END in a single 64 px tall strip, in reading order.
#
# Why: every system that works on this task (UniLipi, HTR-VT, FLAME, the HF CRNNs) feeds the
# network ONE strip of text ~50-96 px tall and reads it left to right. Our earlier runs stacked
# 7 lines into a 448 px tall image, which nobody does - the model had to work out which band each
# character belonged to, on top of reading it. It never got past CER 76.
# This also cuts the blank padding from ~87% of the image to ~61%: far less wasted capacity.
import numpy as np, cv2
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

SH, SW, GAP = CFG['strip_h'], CFG['strip_w'], CFG['line_gap']

def find_lines(g):
    ink = (cv2.GaussianBlur(g, (5, 5), 0) < 128).sum(1).astype(np.float32)
    ink = cv2.GaussianBlur(ink.reshape(-1, 1), (1, 31), 0).ravel()
    on = ink > ink.max() * 0.22
    out, s = [], None
    for i, v in enumerate(on):
        if v and s is None: s = i
        if not v and s is not None:
            if i - s > 12: out.append((s, i))
            s = None
    if s is not None and len(on) - s > 12: out.append((s, len(on)))
    return out

def to_strip(pil):
    g = np.array(pil.convert('L'))
    segs = []
    for y0, y1 in (find_lines(g) or [(0, g.shape[0])]):
        c = g[max(0, y0 - 3):min(g.shape[0], y1 + 3)]
        xs = np.where((c < 128).any(0))[0]
        if len(xs) == 0: continue
        c = c[:, xs[0]:xs[-1] + 1]                       # crop to the ink: no side padding
        segs.append(cv2.resize(c, (max(1, int(c.shape[1] * SH / c.shape[0])), SH)))
    if not segs: segs = [np.full((SH, SH), 255, np.uint8)]
    parts = []
    for i, s in enumerate(segs):
        if i: parts.append(np.full((SH, GAP), 255, np.uint8))   # small gap marks a line break
        parts.append(s)
    strip = np.hstack(parts)
    if strip.shape[1] > SW:                              # too long: scale down to fit
        strip = cv2.resize(strip, (SW, SH))
    canvas = np.full((SH, SW), 255, np.uint8)
    canvas[:, :strip.shape[1]] = strip
    return canvas, len(segs)

CACHE = f"{OUT}/strip_{SH}x{SW}.npy"
if os.path.exists(CACHE):
    IMGS = np.load(CACHE); nline = None
else:
    res = [to_strip(r['image']) for r in tqdm(ds, desc='cutting patches into one strip')]
    IMGS = np.stack([a for a, _ in res]); nline = [b for _, b in res]
    np.save(CACHE, IMGS)
IDX = {n: i for i, n in enumerate(names)}
if nline is not None:
    import collections; print('lines per patch:', dict(sorted(collections.Counter(nline).items())))
print('blank pixels: %.0f%%  (the 7-band stack was ~87%%)' % (100 * (IMGS > 250).mean()))

ex = SPLIT_IDS['train'][0]
plt.figure(figsize=(18, 2)); plt.imshow(IMGS[IDX[ex]][:, :3000], cmap='gray'); plt.axis('off')
plt.title(text[ex][:95]); plt.show()
print('cached', IMGS.shape, f'{IMGS.nbytes/1e9:.2f} GB')

In [ ]:
# Cell 8 - augmentation (training only) + data loaders.
# The transformations are the ones HTR-VT lists: random affine, erosion, dilation, elastic
# distortion and noise. Applied to the whole strip, which is now a single row of text.
from torch.utils.data import Dataset, DataLoader

def augment(img, rng):
    H, W = img.shape
    out = img
    if rng.random() < 0.5:                                   # small rotate / shear / scale
        M = cv2.getRotationMatrix2D((W/2, H/2), rng.uniform(-1.5, 1.5), rng.uniform(0.96, 1.04))
        sh = rng.uniform(-0.10, 0.10); M[0,1] += sh; M[0,2] -= sh*H/2
        M[1,2] += rng.uniform(-2, 2)
        out = cv2.warpAffine(out, M, (W, H), flags=cv2.INTER_LINEAR, borderValue=255)
    if rng.random() < 0.5:                                   # stroke thickness
        k = np.ones((2, 2), np.uint8)
        out = cv2.erode(out, k) if rng.random() < 0.5 else cv2.dilate(out, k)
    if rng.random() < 0.3:                                   # elastic distortion
        a, s = rng.uniform(10, 25), rng.uniform(4, 6)
        dx = cv2.GaussianBlur((rng.random((H, W))*2-1).astype(np.float32), (0,0), s)*a
        dy = cv2.GaussianBlur((rng.random((H, W))*2-1).astype(np.float32), (0,0), s)*a
        gx, gy = np.meshgrid(np.arange(W, dtype=np.float32), np.arange(H, dtype=np.float32))
        out = cv2.remap(out, gx+dx, gy+dy, cv2.INTER_LINEAR, borderValue=255)
    if rng.random() < 0.4:                                   # speckle / contrast
        out = np.clip(out.astype(np.float32)*rng.uniform(0.9,1.1)
                      + rng.normal(0, rng.uniform(0,8), out.shape), 0, 255).astype(np.uint8)
    return out

class DS(Dataset):
    def __init__(s, ids, train, aug): s.ids, s.train, s.aug = ids, train, aug
    def __len__(s): return len(s.ids)
    def __getitem__(s, i):
        n = s.ids[i]; img = IMGS[IDX[n]]
        if s.train and s.aug:
            img = augment(img, np.random.default_rng(torch.randint(0, 2**31-1, (1,)).item()))
        x = torch.from_numpy(np.ascontiguousarray(img)).float().div(255).sub(0.5).div(0.5).unsqueeze(0)
        return x, torch.tensor(encode(text[n]), dtype=torch.long), n

def collate(b):
    xs, ys, ns = zip(*b)
    return torch.stack(xs), torch.cat(ys), torch.tensor([len(y) for y in ys]), list(ns)

def make_loader(ids, train, cfg=None):
    cfg = cfg or CFG; bs, nw = cfg['batch_size'], cfg['num_workers']
    return DataLoader(DS(ids, train, cfg['use_aug']), batch_size=bs, shuffle=train,
                      drop_last=train and len(ids) > bs, num_workers=nw, collate_fn=collate,
                      pin_memory=True, persistent_workers=nw > 0)

rng = np.random.default_rng(0)
fig, ax = plt.subplots(2, 1, figsize=(18, 3))
ax[0].imshow(IMGS[IDX[SPLIT_IDS['train'][1]]][:, :3000], cmap='gray'); ax[0].set_title('original'); ax[0].axis('off')
ax[1].imshow(augment(IMGS[IDX[SPLIT_IDS['train'][1]]], rng)[:, :3000], cmap='gray'); ax[1].set_title('augmented'); ax[1].axis('off')
plt.show(); print('data pipeline ready')

In [ ]:
# Cell 9 - the model. Unchanged from the Colab notebook except that it now reads a band stack.
import torch.nn as nn, torch.nn.functional as F

class ConvBlock(nn.Module):
    def __init__(s, cin, cout, stride, drop=0.1):
        super().__init__()
        s.net = nn.Sequential(nn.Conv2d(cin, cout, 3, padding=1), nn.ReLU(True),
                              nn.Conv2d(cout, cout, 3, padding=1), nn.ReLU(True),
                              nn.Conv2d(cout, cout, 3, stride=stride, padding=1), nn.ReLU(True),
                              nn.InstanceNorm2d(cout, affine=True), nn.Dropout2d(drop))
    def forward(s, x): return s.net(x)

class DSCBlock(nn.Module):
    def __init__(s, c, drop=0.1):
        super().__init__()
        s.net = nn.Sequential(nn.Conv2d(c, c, 3, padding=1, groups=c), nn.Conv2d(c, c, 1), nn.ReLU(True),
                              nn.Conv2d(c, c, 3, padding=1, groups=c), nn.Conv2d(c, c, 1), nn.ReLU(True),
                              nn.InstanceNorm2d(c, affine=True), nn.Dropout2d(drop))
    def forward(s, x): return x + s.net(x)

class Encoder(nn.Module):            # height /64 : 64 px strip -> ONE output row; width /8
    def __init__(s, d=256):
        super().__init__()
        s.net = nn.Sequential(ConvBlock(1,32,(1,1)), ConvBlock(32,64,(2,2)), ConvBlock(64,128,(2,2)),
                              ConvBlock(128,d,(2,2)), ConvBlock(d,d,(4,1)), ConvBlock(d,d,(2,1)),
                              DSCBlock(d), DSCBlock(d), DSCBlock(d), DSCBlock(d))
    def forward(s, x): return s.net(x)

def span_mask(f, ratio=0.1, max_span=8):
    B, D, H, W = f.shape
    m = torch.ones(B, 1, 1, W, device=f.device, dtype=f.dtype)
    for b in range(B):
        for _ in range(max(1, int(W*ratio/((max_span+1)/2)))):
            L = random.randint(1, max_span); st = random.randint(0, W-L); m[b,:,:,st:st+L] = 0
    return f * m

class Reader(nn.Module):
    def __init__(s, n_classes, use_lstm=True, use_count=True, use_span_mask=True, d=256):
        super().__init__()
        s.enc = Encoder(d); s.use_span_mask = use_span_mask
        s.lstm = nn.LSTM(d, d//2, 2, bidirectional=True, batch_first=True, dropout=0.1) if use_lstm else None
        s.head = nn.Conv2d(d, n_classes, 1)
        s.count = nn.Sequential(nn.Linear(d,128), nn.ReLU(True), nn.Linear(128,1)) if use_count else None
    def forward(s, x):
        f = s.enc(x)
        if s.training and s.use_span_mask: f = span_mask(f)
        if s.lstm is not None:
            B, D, H, W = f.shape
            r, _ = s.lstm(f.permute(0,2,3,1).reshape(B*H, W, D))
            f = r.reshape(B,H,W,D).permute(0,3,1,2)
        lo = s.head(f); B, Cc, H, W = lo.shape
        seq = lo.permute(0,2,3,1).reshape(B, H*W, Cc)          # bands top-to-bottom = reading order
        cnt = s.count(f.mean((2,3))).squeeze(1) if s.count is not None else None
        return seq, cnt

DEV = 'cuda'
_m = Reader(NUM_CLASSES, CFG['use_lstm'], CFG['use_count'], CFG['span_mask']).to(DEV).eval()
with torch.no_grad():
    _f = _m.enc(torch.zeros(1,1,CFG['img_h'],CFG['img_w'], device=DEV))
    _s, _ = _m(torch.zeros(1,1,CFG['img_h'],CFG['img_w'], device=DEV))
ROWS, COLS, N_FRAMES = _f.shape[2], _f.shape[3], _s.shape[1]
_mean = np.mean([len(encode(text[n])) for n in names])
print(f'params {sum(p.numel() for p in _m.parameters())/1e6:.1f} M')
print(f'grid {ROWS} row x {COLS} cols = {N_FRAMES} slots | {N_FRAMES/_mean:.1f} slots per character')
assert ROWS == 1, f'{ROWS} output rows - the strip must collapse to a single row of frames'
need = max(len(encode(text[n])) + sum(a==b for a,b in zip(encode(text[n]), encode(text[n])[1:])) for n in names)
assert N_FRAMES >= need, f'not enough slots ({N_FRAMES} < {need})'
del _m, _s, _f; torch.cuda.empty_cache()
print('model OK')

In [ ]:
# Cell 10 - decoding and metrics (unchanged, so numbers stay comparable)
import editdistance, sacrebleu
_FR = ''
_u = lambda s: s.replace('<FRAC>', _FR)

def greedy_decode(seq):
    ids = seq.argmax(-1).cpu().numpy()
    keep = np.ones_like(ids, dtype=bool); keep[:,1:] = ids[:,1:] != ids[:,:-1]; keep &= ids != 0
    return [decode_ids(r[k]) for r, k in zip(ids, keep)]

def compute_metrics(refs, hyps, n_boot=0, seed=0):
    e = np.array([editdistance.eval(_u(r), _u(h)) for r, h in zip(refs, hyps)], float)
    L = np.array([max(1, len(_u(r))) for r in refs], float)
    we = np.array([editdistance.eval(r.split(), h.split()) for r, h in zip(refs, hyps)], float)
    wl = np.array([max(1, len(r.split())) for r in refs], float)
    per = e / L
    out = dict(CER=100*e.sum()/L.sum(), WER=100*we.sum()/wl.sum(),
               median_CER=100*float(np.median(per)), catastrophic_pct=100*float((per>0.5).mean()),
               chrF_pp=sacrebleu.corpus_chrf(hyps, [refs], word_order=2).score,
               BLEU=sacrebleu.corpus_bleu(hyps, [refs]).score, n=len(refs))
    if n_boot:
        rg = np.random.default_rng(seed); idx = rg.integers(0, len(refs), (n_boot, len(refs)))
        b = 100*e[idx].sum(1)/L[idx].sum(1)
        out['CER_CI95'] = (float(np.percentile(b,2.5)), float(np.percentile(b,97.5)))
    return out
print('metrics self-test:', {k: round(v,2) for k,v in compute_metrics(['राम आला'],['राम आल']).items() if k!='n'})

In [ ]:
# Cell 11 - training loop. Changes from the Colab version, all learned the hard way:
#   * runs in FULL fp32. The first Kaggle run trained well for 12 epochs (CER 100 -> 79.6) and
#     then died: weights grew until the fp16 forward pass overflowed, so from epoch 15 every
#     batch produced a non-finite loss and NOTHING trained for the next 45 minutes.
#   * if more than 5% of an epoch's batches are skipped it ABORTS at once instead of burning hours
#   * logs gradient and weight norms so instability is visible before it is fatal
#   * steps the schedule only when the optimiser actually stepped (removes the fp16 warning)
#   * "cold" is judged by LOSS, not CER: a model printing spaces scores CER ~80 and looks busy
#   * obeys the wall-clock budget so Kaggle always commits a finished notebook
CTC = nn.CTCLoss(blank=0, zero_infinity=True)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def loss_fn(seq, cnt, y, yl, cfg):
    logp = seq.float().log_softmax(-1).permute(1,0,2)
    il = torch.full((logp.size(1),), logp.size(0), dtype=torch.long)
    loss = CTC(logp, y, il, yl)
    if cnt is not None:
        loss = loss + cfg['count_weight']*F.l1_loss(cnt.float(), yl.float().to(cnt.device)/100)
    return loss

@torch.no_grad()
def evaluate(model, loader):
    model.eval(); refs, hyps, ns = [], [], []
    for x, y, yl, n in loader:
        seq, _ = model(x.to(DEV, non_blocking=True))
        hyps += greedy_decode(seq); refs += [text[k] for k in n]; ns += n
    return refs, hyps, ns

def fit(cfg, train_ids, val_ids, run_dir, verbose_every=1, stop_at_cer=None):
    os.makedirs(run_dir, exist_ok=True); set_seed(cfg['seed'])
    model = Reader(NUM_CLASSES, cfg['use_lstm'], cfg['use_count'], cfg['span_mask']).to(DEV)
    tl = make_loader(train_ids, True, cfg); vl = make_loader(val_ids, False, cfg)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg['lr'], weight_decay=cfg['weight_decay'])
    upd_per_ep = max(1, len(tl)//cfg['accum'])
    total = max(1, cfg['max_epochs']*upd_per_ep); warm = min(cfg['warmup_steps'], total//10+1)
    if cfg.get('const_lr'):
        sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0,(s+1)/warm))
    else:
        sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s+1)/warm if s < warm else
            max(1e-6/cfg['lr'], 0.5*(1+math.cos(math.pi*(s-warm)/max(1,total-warm)))))
    state = dict(epoch=0, best_cer=1e9, bad_epochs=0)
    last = f'{run_dir}/last.pt'
    if os.path.exists(last):
        ck = torch.load(last, map_location=DEV, weights_only=False)
        model.load_state_dict(ck['model']); opt.load_state_dict(ck['opt'])
        sched.load_state_dict(ck['sched']); state = ck['state']
        print(f"resumed from epoch {state['epoch']} (best val CER {state['best_cer']:.2f})")
    elif cfg.get('init_from') and os.path.exists(cfg['init_from']):
        ck = torch.load(cfg['init_from'], map_location=DEV, weights_only=False)
        model.load_state_dict(ck['model']); print(f"warm-started (its val CER {ck['val']['CER']:.2f})")
    json.dump(cfg, open(f'{run_dir}/config.json','w'), indent=2)
    log = f'{run_dir}/log.csv'
    if not os.path.exists(log):
        csv.writer(open(log,'w',newline='')).writerow(
            ['epoch','train_loss','val_CER','val_WER','lr','sec','skipped','grad_norm','weight_norm'])

    while state['epoch'] < cfg['max_epochs'] and state['bad_epochs'] < cfg['patience']:
        if budget_left() < 0.35:
            print(f'TIME BUDGET reached ({elapsed_h():.1f} h) - stopping so results are saved'); break
        t0 = time.time(); model.train(); tot = nb = skipped = 0; gmax = 0.0
        opt.zero_grad(set_to_none=True)
        for step, (x, y, yl, _) in enumerate(tl):
            seq, cnt = model(x.to(DEV, non_blocking=True))
            loss = loss_fn(seq, cnt, y, yl, cfg)
            if not torch.isfinite(loss):
                opt.zero_grad(set_to_none=True); skipped += 1; continue
            (loss/cfg['accum']).backward()
            if (step+1) % cfg['accum'] == 0:
                g = torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                gmax = max(gmax, float(g)); opt.step(); sched.step()
                opt.zero_grad(set_to_none=True)
            tot += loss.item(); nb += 1
        train_loss = tot/max(1, nb)
        wnorm = float(sum(p.detach().float().pow(2).sum() for p in model.parameters()).sqrt())
        if skipped > 0.05*len(tl):      # the failure that wasted 45 minutes last time
            print(f'ABORTED: {skipped}/{len(tl)} batches gave a non-finite loss - training has '
                  f'destabilised (weight norm {wnorm:,.0f}). Stopping now rather than idling.'); break
        refs, hyps, _ = evaluate(model, vl); m = compute_metrics(refs, hyps)
        state['epoch'] += 1
        cold = train_loss > cfg.get('cold_start_loss', 2.8)
        improved = m['CER'] < state['best_cer'] - cfg['min_delta']
        if improved:
            state['best_cer'] = m['CER']; state['bad_epochs'] = 0
            torch.save({'model':model.state_dict(),'cfg':cfg,'epoch':state['epoch'],'val':m}, f'{run_dir}/best.pt')
        elif cold: state['bad_epochs'] = 0
        else: state['bad_epochs'] += 1
        torch.save({'model':model.state_dict(),'opt':opt.state_dict(),'sched':sched.state_dict(),
                    'state':state}, last)
        lr = opt.param_groups[0]['lr']; sec = time.time()-t0
        csv.writer(open(log,'a',newline='')).writerow(
            [state['epoch'], round(train_loss,4), round(m['CER'],2), round(m['WER'],2), f'{lr:.2e}',
             round(sec), skipped, round(gmax,2), round(wnorm,1)])
        if state['epoch'] % verbose_every == 0 or improved:
            print(f"ep {state['epoch']:3d} | loss {train_loss:6.3f} | val CER {m['CER']:6.2f} | "
                  f"best {state['best_cer']:6.2f} | lr {lr:.1e} | {sec:.0f}s"
                  f" | grad {gmax:5.1f} w {wnorm:6.0f}"
                  f"{' skip'+str(skipped) if skipped else ''}{' *' if improved else ''} | {elapsed_h():.1f}h")
            print('   GT  :', refs[0][:85]); print('   PRED:', hyps[0][:85])
        upd = state['epoch']*upd_per_ep
        # Only give up if it is at the prior AND has shown no real reading at all. A run whose
        # CER is falling is learning, however slowly, and must be left alone.
        if cold and state['best_cer'] > 90 and upd >= cfg.get('cold_start_max_updates', 12000):
            print(f'STOPPED: loss {train_loss:.2f} at the letter-frequency prior and best CER still '
                  f'{state["best_cer"]:.1f} after {upd} updates - it never started reading'); break
        if stop_at_cer is not None and m['CER'] <= stop_at_cer:
            print(f'reached CER {m["CER"]:.2f}, moving on'); break
    print(f"done: {state['epoch']} epochs, best val CER {state['best_cer']:.2f}")
    return model
print('ready')

In [ ]:
# Cell 12 - 8-minute smoke test: can it memorise 20 patches? Proves the pipeline before the long run.
smoke = dict(CFG, run_name='smoke', use_aug=False, span_mask=False, batch_size=2, accum=1,
             const_lr=True, warmup_steps=50, max_epochs=250, patience=250, cold_start_max_updates=99999)
ids20 = SPLIT_IDS['train'][:20]
_ = fit(smoke, ids20, ids20, f'{OUT}/runs/smoke', verbose_every=25, stop_at_cer=5.0)

In [ ]:
# Cell 13 - ONE long, uninterrupted run. No phases, no hand-holding.
#
# Deliberately NOT here any more:
#   * no two-phase split - that was scaffolding for a problem the strip layout removes
#   * no cold-start abort - it stopped a run at epoch 34 of 80 that was still at CER 76
#   * no flat learning rate - that let the model bounce for 23 epochs without settling
#
# What stops it: 25 epochs with no improvement (HF practice is ~20), or the 10.5 h budget.
# Everything else is left alone to train.
model = fit(CFG, SPLIT_IDS['train'], SPLIT_IDS['val'][:150], RUN_DIR, verbose_every=1)

In [ ]:
# Cell 14 - final score on the held-out test split, using the best checkpoint
src = f'{RUN_DIR}/best.pt' if os.path.exists(f'{RUN_DIR}/best.pt') else f'{RUN_DIR}_A/best.pt'
best = torch.load(src, map_location=DEV, weights_only=False)
model = Reader(NUM_CLASSES, CFG['use_lstm'], CFG['use_count'], CFG['span_mask']).to(DEV)
model.load_state_dict(best['model'])
refs, hyps, ns = evaluate(model, make_loader(SPLIT_IDS['test'], False))
res = compute_metrics(refs, hyps, n_boot=10000)
res.update(run=CFG['run_name'], from_checkpoint=src, best_epoch=best['epoch'],
           val_CER=best['val']['CER'], hours=round(elapsed_h(), 2))
json.dump(res, open(f'{OUT}/test_metrics.json','w',encoding='utf-8'), indent=2, ensure_ascii=False)
pd.DataFrame({'image_id':ns,'reference':refs,'prediction':hyps,
              'cer':[editdistance.eval(_u(r),_u(h))/max(1,len(_u(r))) for r,h in zip(refs,hyps)]}
            ).to_csv(f'{OUT}/test_predictions.csv', index=False, encoding='utf-8')
print(json.dumps(res, indent=2, ensure_ascii=False))

In [ ]:
# Cell 15 - curves and example predictions, saved as files you can put in the report
import glob
fig, ax = plt.subplots(1, 2, figsize=(15, 4))
for d, lab in [(RUN_DIR+'_A','phase A'), (RUN_DIR,'phase B')]:
    f = f'{d}/log.csv'
    if not os.path.exists(f): continue
    lg = pd.read_csv(f)
    ax[0].plot(lg.epoch, lg.train_loss, label=lab); ax[1].plot(lg.epoch, lg.val_CER, label=lab)
ax[0].axhline(3.32, ls='--', c='gray'); ax[0].set_title('training loss (dashed = letter-frequency prior)')
ax[1].set_title('validation CER %'); [a.set_xlabel('epoch') or a.legend() or a.grid(alpha=.3) for a in ax]
plt.tight_layout(); plt.savefig(f'{OUT}/curves.png', dpi=130); plt.show()

pred = pd.read_csv(f'{OUT}/test_predictions.csv', keep_default_na=False).sort_values('cer')
for _, r in pd.concat([pred.head(2), pred.tail(2)]).iterrows():
    plt.figure(figsize=(16,5)); plt.imshow(IMGS[IDX[r.image_id]], cmap='gray'); plt.axis('off')
    plt.title(f'{r.image_id}  CER {100*r.cer:.1f}%'); plt.show()
    print('GT  :', r.reference[:120]); print('PRED:', r.prediction[:120]); print()
print('outputs in /kaggle/working :', sorted(os.listdir(OUT)))
print(f'total runtime {elapsed_h():.2f} h')

## Reading the output

**The shape to expect, not steady progress.** Training loss parks near **3.32** — the score of a model
that has learned how often each letter occurs but is ignoring the image — for a while, then drops
sharply. Cell 12 shows this in miniature in about 8 minutes.

| What you see | What it means |
|---|---|
| Cell 12 reaches CER < 5 | pipeline is sound; the long run is worth it |
| Phase A loss falls below ~2.8 | **it has started reading** — this is the moment that matters |
| `STOPPED: loss ... still at the letter-frequency prior` | it never started; send me the log, do not retune |
| `TIME BUDGET reached` | ran out of hours; everything so far is saved, just commit again to continue |

Each run writes `config.json`, `log.csv`, `best.pt` and `last.pt`. **Re-committing resumes** from
`last.pt`, so a second 12-hour run picks up where the first stopped.

### Other seeds and the ablations
Change `run_name` and `seed` in Cell 2 (`B1_seed43`, 43), then run Cells 13–15 again.
For ablations set `use_lstm=False`, `use_count=False` or `use_aug=False` in Cell 2 and give it a new `run_name`.